# 04 Holt's Trend-Corrected Exponential Smoothing

Holt's method, also called double exponential smoothing, adds a trend or growth-rate state to the level state. It is designed for a series with a roughly linear trend and no seasonal pattern.

The update equations are:

$$l_t = \alpha y_t + (1-\alpha)(l_{t-1}+b_{t-1}),$$

$$b_t = \gamma(l_t-l_{t-1}) + (1-\gamma)b_{t-1},$$

and the forecast equation is

$$\hat y_{T+\tau|T} = l_T + \tau b_T.$$

The one-step forecast made before observing period $t$ is

$$\hat y_{t|t-1}=l_{t-1}+b_{t-1},$$

so the one-step forecast error is $e_t=y_t-\hat y_{t|t-1}$.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from checks import check_between, check_close, check_columns
from smoothing_utils import (
    accuracy_measures, initial_level_mean, initial_line,
    simple_es, optimize_simple_es,
    holt_trend, optimize_holt, holt_forecast_table,
    holt_winters, optimize_holt_winters, holt_winters_forecast,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)
DATA_DIR = Path('data')

## Initial Level and Trend from a Straight Line

A common course initialization is to fit a straight line to an early stable portion of the data. If the first $n$ observations are used, fit

$$y_t = l_0 + b_0t + \text{error}, \quad t=1,2,\ldots,n.$$

The intercept becomes the initial level $l_0$, and the slope becomes the initial trend $b_0$. This is exactly what `initial_line(y, n=...)` does.


In [ ]:
therm = pd.read_csv(DATA_DIR / 'thermostat_sales.csv')
y = therm['ThermostatSales'].to_numpy()

first_segment = pd.DataFrame({
    't': np.arange(1, 27),
    'y': y[:26],
})

l0, b0 = initial_line(y, n=26)
print(f'Initial level l0: {l0:.4f}')
print(f'Initial trend b0: {b0:.4f}')
print(check_close(l0, 202.6246, tol=0.001, label='initial level l0 from lecture example'))
print(check_close(b0, -0.3682, tol=0.001, label='initial trend b0 from lecture example'))
first_segment.head()


## Optimizing the Smoothing Constants

For fixed initial values, choose $\alpha$ and $\gamma$ by minimizing the sum of squared one-step forecast errors:

$$SSE(\alpha,\gamma)=\sum_{t=1}^{T}(y_t-\hat y_{t|t-1})^2,$$

subject to

$$0\le \alpha\le 1, \quad 0\le \gamma\le 1.$$

The helper `optimize_holt` performs this constrained numerical search. The resulting table is also the audit trail for the recursion: it contains the observed value, one-step forecast, updated level, updated trend, error, and squared error for each period.


In [ ]:
alpha_opt, gamma_opt, fit_holt, metrics_holt = optimize_holt(y, l0=l0, b0=b0)
print(f'Optimal alpha: {alpha_opt:.4f}')
print(f'Optimal gamma: {gamma_opt:.4f}')
print(check_close(alpha_opt, 0.247, tol=0.005, label='optimized alpha from lecture example'))
print(check_close(gamma_opt, 0.095, tol=0.005, label='optimized gamma from lecture example'))
pd.Series(metrics_holt).to_frame('value')


In [ ]:
recursion_columns = ['Time', 'y', 'yhat_one_step', 'level', 'trend', 'error', 'squared_error']
fit_holt[recursion_columns].head(10).round(3)


The first row starts from the initial values $l_0$ and $b_0$. Each later row uses the previous row's level and trend to make the next one-step forecast, then updates the level and trend after the observed value is known.

When a homework or spreadsheet asks for a recursion table, these are the columns a grader should be able to trace.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(therm['Week'], y, marker='o', label='Observed')
ax.plot(therm['Week'], fit_holt['yhat_one_step'], label='One-step Holt forecast')
ax.plot(therm['Week'], fit_holt['level'], label='Estimated level')
ax.set_xlabel('Week')
ax.set_ylabel('Thermostat sales')
ax.legend()
plt.tight_layout()

## Approximate Prediction Intervals

For the homework-style Holt model, the lecture solution uses an approximate standard error based on the fitted SSE:

$$s = \sqrt{\frac{SSE}{T-2}}.$$

For horizon `tau`, the interval multiplier used in this module is

$$\sqrt{1 + \sum_{j=1}^{\tau-1}\alpha^2(1+j\gamma)^2}.$$

The approximate 95% prediction interval is the point forecast plus or minus `1.96` times `s` times that multiplier. This is a practical interval formula for the course workflow.

In [ ]:
holt_forecast_table(fit_holt, alpha=alpha_opt, gamma=gamma_opt, h=6).round(2)

## Interpretation

The final level tells us the current smoothed level of the series. The final trend tells us the estimated change per period. If the optimized trend smoothing constant is near zero, the model is keeping the trend close to its initial estimate rather than allowing it to change rapidly.